In [ ]:
# --- BLOQUE CORREGIDO ---

from turtle import pd

def clean_experience(x):
    # Si es nulo o vacío, devolvemos 0 (o lo ignoramos)
    if pd.isna(x):
        return 0.0
    if x ==  'More than 30 years':
        return 30.0
    if x == 'Less than 1 year':
        return 0.5
    return float(x)

def clean_education(x):
    # ESTA ES LA LÍNEA NUEVA: Si x no es texto (es un float/NaN), devolvemos "Sin estudios"
    if type(x) != str: 
        return 'Less than a Bachelors'
        
    if 'Bachelor’s degree' in x:
        return 'Bachelor’s degree'
    if 'Master’s degree' in x:
        return 'Master’s degree'
    if 'Professional degree' in x or 'Other doctoral' in x:
        return 'Post grad'
    return 'Less than a Bachelors'

# Antes de aplicar nada, vamos a asegurarnos de limpiar los nulos
# Si hay filas que no tienen ni EdLevel ni Experience, las borramos primero
df = df.dropna(subset=['EdLevel', 'Experience'])

# 1. Aplicamos la limpieza a la Experiencia
df['Experience'] = df['Experience'].apply(clean_experience)

# 2. Aplicamos la limpieza a la Educación
df['EdLevel'] = df['EdLevel'].apply(clean_education)

# 3. Vemos el resultado final
print("✅ ¡Ahora sí! Datos limpios.")
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1. SELECCIÓN FINAL (Para arreglar el warning y limpiar basura)
# Nota: Veo que usaste 'WorkExp' en tu error anterior, así que usaremos esa.
# Asegúrate de que 'ConvertedCompYearly' es el nombre de tu columna de salario (o cámbialo por 'Salary' si ya lo renombraste)
wanted_columns = ["Country", "EdLevel", "WorkExp", "ConvertedCompYearly"]

# Hacemos una copia limpia. Esto elimina el SettingWithCopyWarning
df_final = df[wanted_columns].copy()

# Renombramos para que sea fácil
df_final = df_final.rename(columns={
    "ConvertedCompYearly": "Salary", 
    "WorkExp": "Experience"
})

# 2. FILTRAR PAÍSES (Importante)
# Si un país tiene menos de 50 datos, la IA se confundirá. Los metemos en "Other"
def shorten_categories(categories, cutoff):
    categorical_map = {}
    for i in range(len(categories)):
        if categories.values[i] >= cutoff:
            categorical_map[categories.index[i]] = categories.index[i]
        else:
            categorical_map[categories.index[i]] = 'Other'
    return categorical_map

country_map = shorten_categories(df_final.Country.value_counts(), 400) # Unbral de 400 personas
df_final['Country'] = df_final['Country'].map(country_map)

# 3. TRANSFORMACIÓN A NÚMEROS (Label Encoding)
# Aquí convertimos texto a números únicos
le_country = LabelEncoder()
df_final['Country'] = le_country.fit_transform(df_final['Country'])

le_education = LabelEncoder()
df_final['EdLevel'] = le_education.fit_transform(df_final['EdLevel'])

# 4. RESULTADO LISTO PARA IA
print("✅ Datos listos para entrenar.")
df_final.head()